# TBD Phase 2 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [ ]:
# TODO: Fill this in before submitting.
GROUP_ID = 6
NOTEBOOK_URL = "https://github.com/34Destiny/tbd-workshop-1/blob/master/notebooks/tbd_phase_2_26L.ipynb"
GROUP_MEMBERS = [
    # "Name Surname / student id",
    "Marcin Barej / 331459",
    "Piotr Stępień / 331534",
    "Igor Skiba / 331530",
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [1]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-3.0.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached duckdb-1.5.3-cp313-cp313-win_amd64.whl.metadata (4.2 kB)
  Using cached pyspark-4.1.2-py2.py3-none-any.whl
  Using cached faker-40.19.1-py3-none-any.whl.metadata (16 kB)
  Using cached deltalake-1.6.0-cp310-abi3-win_amd64.whl.metadata (5.5 kB)
  Using cached memory_profiler-0.61.0-py3-none-any.whl.metadata (20 kB)
  Using cached pyarrow-24.0.0-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
  Using cached matplotlib-3.10.9-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached arro3_core-0.8.0-cp311-abi3-win_amd64.whl.metadata (515 bytes)
  Using cached deprecated-1.3.1-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cache


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Marcin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [24]:
import gc
import os
import time
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from memory_profiler import memory_usage
from pyspark.sql import SparkSession

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.13.13
Polars: 1.41.2
Pandas: 3.0.3
DuckDB: 1.5.3
CPU logical cores: 16
RAM GiB: 31.83


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


## Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [25]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags", "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates", "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'Game events',
 'feature': 'player sessions',
 'stress': 'high-cardinality group-by'}

## Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [26]:
# TODO: Choose the main dataset scale for your final benchmark and verify output paths before generation.
# N_ROWS is the main row count reported for this notebook. Extra row counts are optional stress tests.
# Dataset configuration
SCALE = "small"
SCALE_ROWS = {
    "debug": 200_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("../data/phase2_26L") / f"group_{GROUP_ID:02d}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

# Required negative baseline paths for the file-format/layout task. Do not commit these generated files.
CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

# Leave SEED as None if you want independent data on each generation.
# If you need to reproduce exactly the same dataset later, set SEED to the value stored in the manifest.
SEED = None
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 6 {'name': 'Game events', 'feature': 'player sessions', 'stress': 'high-cardinality group-by'}
Rows: 2000000
Run seed recorded in manifest: 233292846413964125202309585453006053487
Output directory: ..\data\phase2_26L\group_06


## Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [27]:
# TODO: Adapt customize_for_variant(...) and generate_dimension_table(...) to your variant.
def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))


def customize_for_variant(df, card, rng):
    n = len(df)

    df = df.rename({"entity_id": "player_id"})

    game_ids = rng.integers(1, 201, size=n)
    df = df.with_columns(pl.Series("game_id", game_ids))

    session_ids = (df["player_id"].to_numpy() * 23 + rng.integers(0, 20, size=n)) % 1_000_000 + 1
    df = df.with_columns(pl.Series("session_id", session_ids))

    event_types = rng.choice(
        ["kill", "move", "purchase", "achievement", "death", "login", "logout"],
        size=n,
        p=[0.30, 0.35, 0.05, 0.05, 0.15, 0.05, 0.05],
    )
    df = df.with_columns(pl.Series("event_type", event_types))

    score_gained = rng.lognormal(mean=2.5, sigma=1.2, size=n).round(0).astype(int)
    zero_mask = np.isin(event_types, ["login", "logout", "move"])
    score_gained[zero_mask] = 0
    df = df.with_columns(pl.Series("score_gained", score_gained.astype(np.int64)))

    levels = np.clip(rng.exponential(scale=15, size=n).astype(int) + 1, 1, 100)
    df = df.with_columns(pl.Series("level", levels.astype(np.int64)))

    session_dur = rng.integers(30, 7_201, size=n).astype(np.float64)
    session_dur[~np.isin(event_types, ["login"])] = np.nan
    df = df.with_columns(pl.Series("session_duration_s", session_dur))

    return df


def generate_dimension_table(card, rng):
    n_games = 200
    genres = rng.choice(
        ["action", "rpg", "strategy", "sports", "puzzle", "simulation", "horror"],
        size=n_games,
    )
    platforms = rng.choice(
        ["PC", "console", "mobile", "cross-platform"],
        size=n_games,
        p=[0.35, 0.30, 0.25, 0.10],
    )
    avg_player_rating = np.clip(rng.normal(loc=7.0, scale=1.5, size=n_games), 1.0, 10.0).round(1)
    release_year = rng.integers(2015, 2026, size=n_games)
    return pl.DataFrame(
        {
            "game_id": np.arange(1, n_games + 1),
            "genre": genres,
            "platform": platforms,
            "avg_player_rating": avg_player_rating,
            "release_year": release_year,
        }
    )

In [28]:
# TODO: Run this after adapting the generator. Verify that generated data is not committed to Git.
# Generate and save the dataset
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)
dimension = generate_dimension_table(CARD, rng)

events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

# Optional partitioned layout for experiments with predicate pushdown and file layout.
events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

# TODO: Create an optimized Parquet layout for one selected query pattern.
# Example ideas:
# - sort by columns used in range filters before writing,
# - choose a smaller row_group_size if it improves row-group pruning,
# - partition by date or another selective filter column,
# - add bloom filters only if your chosen writer and reader expose this option clearly.
# Replace the sort columns with columns from your own query pattern.
# events.sort(["event_date", "category"]).write_parquet(
#     OPTIMIZED_EVENTS_PATH,
#     compression="zstd",
#     row_group_size=100_000,
# )

events.sort(["event_type", "game_id", "player_id"]).write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=50_000,
)
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events": str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized": str(OPTIMIZED_EVENTS_PATH),
        "dimension": str(DIMENSION_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "polars": pl.__version__,
        "pandas": pd.__version__,
        "duckdb": duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))


{
  "created_at_utc": "2026-06-09T16:29:29.383395+00:00",
  "group_id": 6,
  "variant": {
    "name": "Game events",
    "feature": "player sessions",
    "stress": "high-cardinality group-by"
  },
  "scale": "small",
  "rows": 2000000,
  "run_seed": 233292846413964125202309585453006053487,
  "paths": {
    "events": "..\\data\\phase2_26L\\group_06\\events.parquet",
    "events_partitioned": "..\\data\\phase2_26L\\group_06\\events_partitioned",
    "events_optimized": "..\\data\\phase2_26L\\group_06\\events_optimized.parquet",
    "dimension": "..\\data\\phase2_26L\\group_06\\dimension.parquet"
  },
  "environment": {
    "python": "3.13.13",
    "polars": "1.41.2",
    "pandas": "3.0.3",
    "duckdb": "1.5.3",
    "cpu_logical_cores": 16,
    "ram_gib": 31.83
  }
}


## Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [29]:
# TODO: Inspect schema, row count, null counts, and basic category distributions.
# Keep this section short, but include enough evidence that your data was generated correctly.

print("Schema")
print(events.schema)
print(f"\n Rows: {events.height:,}  |  Columns: {events.width}")

print("\n Null counts")
print(events.null_count())

print("\n event_type distribution")
print(
    events.group_by("event_type")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .with_columns((pl.col("count") / events.height * 100).round(1).alias("pct_%"))
)

print("\n device distribution")
print(
    events.group_by("device")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print("\n country distribution")
print(
    events.group_by("country")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

n_players = events["player_id"].n_unique()
hot_threshold = int(200_000 * 0.02)
hot_share = (
    events.filter(pl.col("player_id") <= hot_threshold).height / events.height * 100
)
print(f"\n Player skew")
print(f"Unique players: {n_players:,}")
print(f"Top 2% players (id <= {hot_threshold}) hold {hot_share:.1f}% of events (expected ~50%)")

print("\n Numeric stats")
print(events.select(["score_gained", "level", "session_duration_s", "metric_1", "metric_2"]).describe())

print("\n Dimension table")
print(dimension.schema)
print(f"Games: {dimension.height}")
print(dimension.group_by("genre").agg(pl.len().alias("count")).sort("count", descending=True))
print(dimension.group_by("platform").agg(pl.len().alias("count")).sort("count", descending=True))


Schema
Schema({'event_id': Int64, 'player_id': Int64, 'event_ts': Datetime(time_unit='us', time_zone=None), 'category': String, 'country': String, 'device': String, 'metric_1': Float64, 'metric_2': Int64, 'tags': List(String), 'event_date': Date, 'game_id': Int64, 'session_id': Int64, 'event_type': String, 'score_gained': Int64, 'level': Int64, 'session_duration_s': Float64})

 Rows: 2,000,000  |  Columns: 16

 Null counts
shape: (1, 16)
┌──────────┬───────────┬──────────┬──────────┬───┬────────────┬──────────────┬───────┬─────────────┐
│ event_id ┆ player_id ┆ event_ts ┆ category ┆ … ┆ event_type ┆ score_gained ┆ level ┆ session_dur │
│ ---      ┆ ---       ┆ ---      ┆ ---      ┆   ┆ ---        ┆ ---          ┆ ---   ┆ ation_s     │
│ u32      ┆ u32       ┆ u32      ┆ u32      ┆   ┆ u32        ┆ u32          ┆ u32   ┆ ---         │
│          ┆           ┆          ┆          ┆   ┆            ┆              ┆       ┆ u32         │
╞══════════╪═══════════╪══════════╪══════════╪═══╪═══

## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [30]:
import gc, time
from memory_profiler import memory_usage

# TODO: Implement or adapt benchmark helpers before collecting final results.
BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

benchmark_results = []

# TODO: Implement your timing and memory measurement helper.
# Suggested output: one dictionary matching BENCHMARK_COLUMNS per library/engine/query/mode.
# Recommended inside each measured repetition:
# gc.collect()
# start = time.perf_counter()

def get_file_size_mb(path):
    if path is None:
        return 0.0

    if isinstance(path, (list, tuple)):
        return sum(get_file_size_mb(p) for p in path)

    p = Path(path)

    if p.is_dir():
        return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024 ** 2

    return p.stat().st_size / 1024 ** 2 if p.exists() else 0.0

def run_benchmark(fn, *, library_engine, mode, query_name, data_format="parquet", layout="flat", rows, input_path=None, result_check=None, notes="", n_reps=3):
    times = []
    memory_peaks = []
    last_result = None

    for _ in range(n_reps):
        gc.collect()

        def measured_fn():
            start = time.perf_counter()
            result = fn()
            elapsed = time.perf_counter() - start
            return elapsed, result

        mem_trace, returned = memory_usage(
            measured_fn,
            retval=True,
            interval=0.05,
            max_usage=False,
            include_children=True,
        )

        elapsed, result = returned
        times.append(elapsed)
        memory_peaks.append(max(mem_trace) - min(mem_trace))
        last_result = result

    median_t = round(float(np.median(times)), 4)

    record = {
        "library_engine": library_engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": rows,
        "median_time_s": median_t,
        "peak_memory_mb": round(float(np.median(memory_peaks)), 2),
        "input_size_mb": round(get_file_size_mb(input_path), 2),
        "result_check": result_check,
        "notes": notes,
    }

    benchmark_results.append(record)

    print(
        f"[{library_engine:15s}] {query_name:40s} | "
        f"{median_t:.4f}s | "
        f"mem +{record['peak_memory_mb']:.2f} MB | "
        f"input {record['input_size_mb']:.2f} MB"
    )

    return record, last_result


## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


In [31]:
QUERY_SPECS = [
    {
        "query_name": "top_players_by_score",
        "intent": (
            "Wyszukanie 100 najlepszych graczy pod kątem sumy punktów ze zdarzeń typu 'kill' i 'achievement'. "
            "Testuje wydajność grupowania po kolumnie o wysokiej kardynalności (player_id, około 200k unikalnych wartości) "
            "z selektywnym filtrem na event_type. Główny test obciążeniowy dla wariantu."
        ),
        "stress_tested": "grupowanie o wysokiej kardynalności, filtr selektywny, sortowanie top-k",
        "expected_result": "100 wierszy: player_id, total_score",
        "sql_equivalent": """
            SELECT player_id, SUM(score_gained) AS total_score
            FROM events
            WHERE event_type IN ('kill', 'achievement')
            GROUP BY player_id
            ORDER BY total_score DESC
            LIMIT 100
        """,
    },

    {
        "query_name": "avg_session_duration_by_game_genre",
        "intent": (
            "Złączenie tabeli zdarzeń z tabelą słownikową gier (game_id) i wyliczenie średniego czasu trwania sesji "
            "dla poszczególnych gatunków gier (pomijając null i NaN). "
            "Testuje klasyczny join typu small-build / large-probe (tabela słownikowa ma tylko 200 wierszy) "
            "oraz agregację z obsługą wartości pustych."
        ),
        "stress_tested": "złączenie typu wiele-do-jednego, agregacja z obsługą nulli i NaN, mała tabela słownikowa",
        "expected_result": "7 wierszy: genre, avg_session_duration_s",
        "sql_equivalent": """
            SELECT g.genre, AVG(e.session_duration_s) AS avg_session_duration_s
            FROM events e
            JOIN games g ON e.game_id = g.game_id
            WHERE e.session_duration_s IS NOT NULL
            AND NOT isnan(e.session_duration_s)
            GROUP BY g.genre
            ORDER BY avg_session_duration_s DESC
        """,
    },
    
    {
        "query_name": "daily_active_players_per_country",
        "intent": (
            "Wyznaczenie liczby unikalnych graczy (DAU) dla każdego dnia i kraju w I kwartale 2026 roku. "
            "Testuje grupowanie po dwóch kluczach (data i kraj) połączone z operacją COUNT DISTINCT. "
            "To obciąża pamięć silników bazodanowych przez konieczność budowania dużych struktur hashujących dla unikalnych wartości."
        ),
        "stress_tested": "COUNT DISTINCT, grupowanie po dwóch kluczach, pełny skan tabeli w zakresie dat",
        "expected_result": "około 630 wierszy (maks. 90 dni x 7 krajów): event_date, country, dau",
        "sql_equivalent": """
            SELECT event_date, country, COUNT(DISTINCT player_id) AS dau
            FROM events
            GROUP BY event_date, country
            ORDER BY event_date, country
        """,
    },
]

for i, q in enumerate(QUERY_SPECS, 1):
    print(f"Q{i}: {q['query_name']}")
    print(f"Intent : {q['intent'][:120]}...")
    print(f"Stresses : {q['stress_tested']}")
    print(f"Result : {q['expected_result']}")
    print()


Q1: top_players_by_score
Intent : Wyszukanie 100 najlepszych graczy pod kątem sumy punktów ze zdarzeń typu 'kill' i 'achievement'. Testuje wydajność grupo...
Stresses : grupowanie o wysokiej kardynalności, filtr selektywny, sortowanie top-k
Result : 100 wierszy: player_id, total_score

Q2: avg_session_duration_by_game_genre
Intent : Złączenie tabeli zdarzeń z tabelą słownikową gier (game_id) i wyliczenie średniego czasu trwania sesji dla poszczególnyc...
Stresses : złączenie typu wiele-do-jednego, agregacja z obsługą nulli i NaN, mała tabela słownikowa
Result : 7 wierszy: genre, avg_session_duration_s

Q3: daily_active_players_per_country
Intent : Wyznaczenie liczby unikalnych graczy (DAU) dla każdego dnia i kraju w I kwartale 2026 roku. Testuje grupowanie po dwóch ...
Stresses : COUNT DISTINCT, grupowanie po dwóch kluczach, pełny skan tabeli w zakresie dat
Result : około 630 wierszy (maks. 90 dni x 7 krajów): event_date, country, dau



### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


In [ ]:
# TODO: Configure Spark local only when you start the PySpark local benchmark.
# Initialize Spark only when you start the Spark part of the benchmark.
# TODO: Adjust memory and local core count if needed.

# spark = (
#     SparkSession.builder
#     .appName("TBDPhase2LocalBenchmark")
#     .master("local[*]")
#     .config("spark.driver.memory", "4g")
#     .getOrCreate()
# )

spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "8g") 
    .getOrCreate()
)



In [32]:
# TODO: Pandas implementations of your three queries.
# Implement both Pandas read variants:
# 1. default backend: pd.read_parquet(path)
# 2. PyArrow backend: pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")
#
# Report dtypes for both variants and compare runtime/memory.

def q1_pd(df):
    return (df[df["event_type"].isin(["kill", "achievement"])].groupby("player_id")["score_gained"].sum().nlargest(100))

def q2_pd(df, dim):
    return (df[df["session_duration_s"].notna() & np.isfinite(df["session_duration_s"])].merge(dim[["game_id", "genre"]], on="game_id").groupby("genre")["session_duration_s"].mean().sort_values(ascending=False))

def q3_pd(df):
    return df.groupby(["event_date", "country"])["player_id"].nunique()


#NumPy
df_np  = pd.read_parquet(EVENTS_PATH)
dim_np = pd.read_parquet(DIMENSION_PATH)
print("NumPy dtypes:\n", df_np.dtypes)

input_path=EVENTS_PATH
run_benchmark(lambda: q1_pd(df_np), library_engine="pandas-numpy",  mode="eager", query_name="top_players_by_score", rows=df_np.shape[0], result_check=100, input_path=EVENTS_PATH)

input_path=[EVENTS_PATH, DIMENSION_PATH]
run_benchmark(lambda: q2_pd(df_np, dim_np), library_engine="pandas-numpy", mode="eager", query_name="avg_session_duration_by_game_genre", rows=df_np.shape[0], result_check=7, input_path=[EVENTS_PATH, DIMENSION_PATH])

input_path=EVENTS_PATH
run_benchmark(lambda: q3_pd(df_np), library_engine="pandas-numpy", mode="eager", query_name="daily_active_players_per_country", rows=df_np.shape[0], input_path=EVENTS_PATH)

del df_np, dim_np

#PyArrow
df_pa  = pd.read_parquet(EVENTS_PATH,    engine="pyarrow", dtype_backend="pyarrow")
dim_pa = pd.read_parquet(DIMENSION_PATH, engine="pyarrow", dtype_backend="pyarrow")
print("PyArrow dtypes:\n", df_pa.dtypes)

run_benchmark(lambda: q1_pd(df_pa), library_engine="pandas-pyarrow", mode="eager", query_name="top_players_by_score", rows=df_pa.shape[0], result_check=100, input_path=EVENTS_PATH)

run_benchmark(lambda: q2_pd(df_pa, dim_pa), library_engine="pandas-pyarrow", mode="eager", query_name="avg_session_duration_by_game_genre", rows=df_pa.shape[0], result_check=7, input_path=[EVENTS_PATH, DIMENSION_PATH])

run_benchmark(lambda: q3_pd(df_pa), library_engine="pandas-pyarrow", mode="eager", query_name="daily_active_players_per_country", rows=df_pa.shape[0], input_path=EVENTS_PATH)

del df_pa, dim_pa



NumPy dtypes:
 event_id                       int64
player_id                      int64
event_ts              datetime64[us]
category                         str
country                          str
device                           str
metric_1                     float64
metric_2                       int64
tags                          object
event_date                    object
game_id                        int64
session_id                     int64
event_type                       str
score_gained                   int64
level                          int64
session_duration_s           float64
dtype: object
[pandas-numpy   ] top_players_by_score                     | 0.1807s | mem +127.05 MB | input 51.36 MB
[pandas-numpy   ] avg_session_duration_by_game_genre       | 0.0506s | mem +65.34 MB | input 51.37 MB
[pandas-numpy   ] daily_active_players_per_country         | 0.3263s | mem +205.34 MB | input 51.36 MB
PyArrow dtypes:
 event_id                                          int6

In [33]:
# TODO: Polars implementations of your three queries.
# Required modes:
# - eager: read_parquet -> transformations
# - lazy default: scan_parquet -> transformations -> collect()
# - lazy streaming: scan_parquet -> transformations -> collect(engine="streaming")

def q1_pl(df):
    return (df.filter(pl.col("event_type").is_in(["kill", "achievement"])).group_by("player_id").agg(pl.col("score_gained").sum().alias("total_score")).sort("total_score", descending=True).head(100))
def q2_pl(df, dim):
    return (df.filter(pl.col("session_duration_s").is_not_null() & pl.col("session_duration_s").is_not_nan()).join(dim.select(["game_id", "genre"]), on="game_id", how="inner").group_by("genre").agg(pl.col("session_duration_s").mean().alias("avg_session_duration_s")).sort("avg_session_duration_s", descending=True))
def q3_pl(df):
    return (df.group_by(["event_date", "country"]).agg(pl.col("player_id").n_unique().alias("dau")).sort(["event_date", "country"]))

df_pl  = pl.read_parquet(EVENTS_PATH)

dim_pl = pl.read_parquet(DIMENSION_PATH)

run_benchmark(lambda: q1_pl(df_pl), library_engine="polars", mode="eager", query_name="top_players_by_score", rows=df_pl.height, result_check=100, input_path=EVENTS_PATH)

run_benchmark(lambda: q2_pl(df_pl, dim_pl), library_engine="polars", mode="eager", query_name="avg_session_duration_by_game_genre", rows=df_pl.height, result_check=7, input_path=[EVENTS_PATH, DIMENSION_PATH])

run_benchmark(lambda: q3_pl(df_pl), library_engine="polars", mode="eager", query_name="daily_active_players_per_country", rows=df_pl.height, input_path=EVENTS_PATH)

del df_pl, dim_pl
# lazy default
N = pl.scan_parquet(EVENTS_PATH).select(pl.len()).collect().item()

run_benchmark(lambda: q1_pl(pl.scan_parquet(EVENTS_PATH)).collect(), library_engine="polars", mode="lazy", query_name="top_players_by_score", rows=N, result_check=100, input_path=EVENTS_PATH)

run_benchmark(lambda: q2_pl(pl.scan_parquet(EVENTS_PATH), pl.scan_parquet(DIMENSION_PATH)).collect(), library_engine="polars", mode="lazy", query_name="avg_session_duration_by_game_genre", rows=N, result_check=7, input_path=[EVENTS_PATH, DIMENSION_PATH])

run_benchmark(lambda: q3_pl(pl.scan_parquet(EVENTS_PATH)).collect(), library_engine="polars", mode="lazy", query_name="daily_active_players_per_country", rows=N, input_path=EVENTS_PATH)

# lazy streaming
run_benchmark(lambda: q1_pl(pl.scan_parquet(EVENTS_PATH)).collect(engine="streaming"), library_engine="polars", mode="streaming", query_name="top_players_by_score", rows=N, result_check=100, input_path=EVENTS_PATH)

run_benchmark(lambda: q2_pl(pl.scan_parquet(EVENTS_PATH), pl.scan_parquet(DIMENSION_PATH)).collect(engine="streaming"), library_engine="polars", mode="streaming", query_name="avg_session_duration_by_game_genre", rows=N, result_check=7, input_path=[EVENTS_PATH, DIMENSION_PATH])

run_benchmark(lambda: q3_pl(pl.scan_parquet(EVENTS_PATH)).collect(engine="streaming"), library_engine="polars", mode="streaming", query_name="daily_active_players_per_country", rows=N, input_path=EVENTS_PATH)

[polars         ] top_players_by_score                     | 0.0514s | mem +64.69 MB | input 51.36 MB
[polars         ] avg_session_duration_by_game_genre       | 0.0269s | mem +5.17 MB | input 51.37 MB
[polars         ] daily_active_players_per_country         | 0.0684s | mem +145.13 MB | input 51.36 MB
[polars         ] top_players_by_score                     | 0.0304s | mem +46.78 MB | input 51.36 MB
[polars         ] avg_session_duration_by_game_genre       | 0.0083s | mem +3.16 MB | input 51.37 MB
[polars         ] daily_active_players_per_country         | 0.0479s | mem +111.00 MB | input 51.36 MB
[polars         ] top_players_by_score                     | 0.0211s | mem +22.91 MB | input 51.36 MB
[polars         ] avg_session_duration_by_game_genre       | 0.0079s | mem +2.48 MB | input 51.37 MB
[polars         ] daily_active_players_per_country         | 0.0800s | mem +131.53 MB | input 51.36 MB


({'library_engine': 'polars',
  'mode': 'streaming',
  'query_name': 'daily_active_players_per_country',
  'data_format': 'parquet',
  'layout': 'flat',
  'rows': 2000000,
  'median_time_s': 0.08,
  'peak_memory_mb': 131.53,
  'input_size_mb': 51.36,
  'result_check': None,
  'notes': ''},
 shape: (630, 3)
 ┌────────────┬─────────┬──────┐
 │ event_date ┆ country ┆ dau  │
 │ ---        ┆ ---     ┆ ---  │
 │ date       ┆ str     ┆ u32  │
 ╞════════════╪═════════╪══════╡
 │ 2026-01-01 ┆ BR      ┆ 2918 │
 │ 2026-01-01 ┆ DE      ┆ 2812 │
 │ 2026-01-01 ┆ FR      ┆ 2885 │
 │ 2026-01-01 ┆ IN      ┆ 2870 │
 │ 2026-01-01 ┆ PL      ┆ 2990 │
 │ …          ┆ …       ┆ …    │
 │ 2026-03-31 ┆ FR      ┆ 2889 │
 │ 2026-03-31 ┆ IN      ┆ 2873 │
 │ 2026-03-31 ┆ PL      ┆ 2944 │
 │ 2026-03-31 ┆ UK      ┆ 2900 │
 │ 2026-03-31 ┆ US      ┆ 2945 │
 └────────────┴─────────┴──────┘)

In [34]:
# TODO: DuckDB SQL implementations of your three queries.
# Consider querying Parquet files directly instead of first loading all data into Pandas.

con = duckdb.connect()
def q1_ddb():
    return con.execute(f"""
        SELECT player_id, SUM(score_gained) AS total_score
        FROM read_parquet('{EVENTS_PATH}')
        WHERE event_type IN ('kill', 'achievement')
        GROUP BY player_id
        ORDER BY total_score DESC
        LIMIT 100
    """).df()

def q2_ddb():
    return con.execute(f"""
        SELECT g.genre, AVG(e.session_duration_s) AS avg_session_duration_s
        FROM read_parquet('{EVENTS_PATH}') e
        JOIN read_parquet('{DIMENSION_PATH}') g ON e.game_id = g.game_id
        WHERE e.session_duration_s IS NOT NULL
        AND NOT isnan(e.session_duration_s)
        GROUP BY g.genre
        ORDER BY avg_session_duration_s DESC
    """).df()

def q3_ddb():
    return con.execute(f"""
        SELECT event_date, country, COUNT(DISTINCT player_id) AS dau
        FROM read_parquet('{EVENTS_PATH}')
        GROUP BY event_date, country
        ORDER BY event_date, country
    """).df()

N = con.execute(f"SELECT COUNT(*) FROM read_parquet('{EVENTS_PATH}')").fetchone()[0]
run_benchmark(q1_ddb, library_engine="duckdb", mode="eager", query_name="top_players_by_score", rows=N, result_check=100, input_path=EVENTS_PATH)
run_benchmark(q2_ddb, library_engine="duckdb", mode="eager", query_name="avg_session_duration_by_game_genre", rows=N, result_check=7, input_path=[EVENTS_PATH, DIMENSION_PATH])
run_benchmark(q3_ddb, library_engine="duckdb", mode="eager", query_name="daily_active_players_per_country", rows=N, input_path=EVENTS_PATH)
con.close()

[duckdb         ] top_players_by_score                     | 0.0295s | mem +42.09 MB | input 51.36 MB
[duckdb         ] avg_session_duration_by_game_genre       | 0.0154s | mem +27.13 MB | input 51.37 MB
[duckdb         ] daily_active_players_per_country         | 0.0595s | mem +158.90 MB | input 51.36 MB


In [35]:
# TODO: PySpark local implementations of your three queries.

df_spark  = spark.read.parquet(str(EVENTS_PATH))

dim_spark = spark.read.parquet(str(DIMENSION_PATH))

df_spark.createOrReplaceTempView("events")

dim_spark.createOrReplaceTempView("games")

def q1_spark():
    return spark.sql("""
        SELECT player_id, SUM(score_gained) AS total_score
        FROM events
        WHERE event_type IN ('kill', 'achievement')
        GROUP BY player_id
        ORDER BY total_score DESC
        LIMIT 100
    """).collect()

def q2_spark():
    return spark.sql("""
        SELECT g.genre, AVG(e.session_duration_s) AS avg_session_duration_s
        FROM events e
        JOIN games g ON e.game_id = g.game_id
        WHERE e.session_duration_s IS NOT NULL
        AND NOT isnan(e.session_duration_s)
        GROUP BY g.genre
        ORDER BY avg_session_duration_s DESC
    """).collect()

def q3_spark():
    return spark.sql("""
        SELECT event_date, country, COUNT(DISTINCT player_id) AS dau
        FROM events
        GROUP BY event_date, country
        ORDER BY event_date, country
    """).collect()

N = df_spark.count()

run_benchmark(q1_spark, library_engine="pyspark", mode="lazy", query_name="top_players_by_score", rows=N, result_check=100, input_path=EVENTS_PATH)

run_benchmark(q2_spark, library_engine="pyspark", mode="lazy", query_name="avg_session_duration_by_game_genre", rows=N, result_check=7, input_path=[EVENTS_PATH, DIMENSION_PATH])

run_benchmark(q3_spark, library_engine="pyspark", mode="lazy", query_name="daily_active_players_per_country", rows=N, input_path=EVENTS_PATH)

[pyspark        ] top_players_by_score                     | 2.0197s | mem +187.30 MB | input 51.36 MB
[pyspark        ] avg_session_duration_by_game_genre       | 0.1626s | mem +68.41 MB | input 51.37 MB
[pyspark        ] daily_active_players_per_country         | 5.2686s | mem +72.75 MB | input 51.36 MB


({'library_engine': 'pyspark',
  'mode': 'lazy',
  'query_name': 'daily_active_players_per_country',
  'data_format': 'parquet',
  'layout': 'flat',
  'rows': 2000000,
  'median_time_s': 5.2686,
  'peak_memory_mb': 72.75,
  'input_size_mb': 51.36,
  'result_check': None,
  'notes': ''},
 [Row(event_date=datetime.date(2026, 1, 1), country='BR', dau=2918),
  Row(event_date=datetime.date(2026, 1, 1), country='DE', dau=2812),
  Row(event_date=datetime.date(2026, 1, 1), country='FR', dau=2885),
  Row(event_date=datetime.date(2026, 1, 1), country='IN', dau=2870),
  Row(event_date=datetime.date(2026, 1, 1), country='PL', dau=2990),
  Row(event_date=datetime.date(2026, 1, 1), country='UK', dau=2815),
  Row(event_date=datetime.date(2026, 1, 1), country='US', dau=2879),
  Row(event_date=datetime.date(2026, 1, 2), country='BR', dau=2914),
  Row(event_date=datetime.date(2026, 1, 2), country='DE', dau=2977),
  Row(event_date=datetime.date(2026, 1, 2), country='FR', dau=2969),
  Row(event_date=datet

### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [36]:
# TODO 2.5: Build and benchmark one optimized layout for one selected query.
# Suggested steps:
# 1. Choose one query with a selective filter or column subset.
# 2. Write a baseline Parquet file/directory.
# 3. Write an optimized Parquet file/directory, e.g. sorted and with a selected row_group_size.
# 4. Write CSV or JSONL as a required negative baseline.
#    If your full dataset has nested/list columns, write a flat query-specific baseline with the columns needed by the selected query.
# 5. Benchmark the same logical query on default Parquet, optimized Parquet, and CSV/JSONL.
# 6. Record IO/pruning evidence where available.

# YOUR CODE HERE

events_flat = events.select(["player_id", "event_type", "score_gained"])
events_flat.write_csv(CSV_EVENTS_PATH)

size_default = get_file_size_mb(EVENTS_PATH)
size_optimized = get_file_size_mb(OPTIMIZED_EVENTS_PATH)
size_csv = get_file_size_mb(CSV_EVENTS_PATH)
print(f"Rozmiar pliku Parquet (domyślny): {size_default:.2f} MB")
print(f"Rozmiar pliku Parquet (zoptymalizowany pod Q1 - sortowany): {size_optimized:.2f} MB")
print(f"Rozmiar płaskiego pliku CSV (baseline): {size_csv:.2f} MB")

def run_q1_default():
    return (pl.scan_parquet(EVENTS_PATH).filter(pl.col("event_type").is_in(["kill", "achievement"])).group_by("player_id").agg(pl.col("score_gained").sum().alias("total_score")).sort("total_score", descending=True).head(100).collect())

def run_q1_optimized():
    return (pl.scan_parquet(OPTIMIZED_EVENTS_PATH).filter(pl.col("event_type").is_in(["kill", "achievement"])).group_by("player_id").agg(pl.col("score_gained").sum().alias("total_score")).sort("total_score", descending=True).head(100).collect())

def run_q1_csv():
    return (pl.scan_csv(CSV_EVENTS_PATH).filter(pl.col("event_type").is_in(["kill", "achievement"])).group_by("player_id").agg(pl.col("score_gained").sum().alias("total_score")).sort("total_score", descending=True).head(100).collect())

res_def, q1_layout_default = run_benchmark(
    run_q1_default,
    library_engine="polars-lazy",
    mode="lazy",
    query_name="q1_layout_default",
    data_format="parquet",
    layout="default",
    rows=N_ROWS,
    input_path=EVENTS_PATH,
    result_check=100,
)

res_opt, q1_layout_optimized = run_benchmark(
    run_q1_optimized,
    library_engine="polars-lazy",
    mode="lazy",
    query_name="q1_layout_optimized",
    data_format="parquet",
    layout="optimized",
    rows=N_ROWS,
    input_path=OPTIMIZED_EVENTS_PATH,
    result_check=100,
)

res_csv, q1_layout_csv = run_benchmark(
    run_q1_csv,
    library_engine="polars-lazy",
    mode="lazy",
    query_name="q1_layout_csv",
    data_format="csv",
    layout="flat",
    rows=N_ROWS,
    input_path=CSV_EVENTS_PATH,
    result_check=100,
)





Rozmiar pliku Parquet (domyślny): 51.36 MB
Rozmiar pliku Parquet (zoptymalizowany pod Q1 - sortowany): 51.02 MB
Rozmiar płaskiego pliku CSV (baseline): 26.31 MB
[polars-lazy    ] q1_layout_default                        | 0.0285s | mem +36.70 MB | input 51.36 MB
[polars-lazy    ] q1_layout_optimized                      | 0.0229s | mem +6.23 MB | input 51.02 MB
[polars-lazy    ] q1_layout_csv                            | 0.0512s | mem +98.39 MB | input 26.31 MB


### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [37]:
# TODO 3.1: Implement Polars execution-mode experiments.
#
# Required variants:
# 1. eager: read_parquet -> filter/transform
# 2. lazy: scan_parquet -> filter/transform -> collect()
# 3. streaming collect: scan_parquet -> filter/transform -> collect(engine="streaming")
# 4. streaming sink: scan_parquet -> filter/transform -> sink_parquet(...)
#
# Recommended:
# - use a query whose output has many rows, not a tiny aggregate table,
# - measure each mode in a fresh process if possible,
# - call gc.collect() before each measured run,
# - record runtime, peak memory, output row count, and output size,
# - append results to benchmark_results.

# YOUR CODE HERE

QUERY_OUTPUT_PATH = OUTPUT_DIR / "q3_1_output.parquet"

def eager_variant():
    df = pl.read_parquet(EVENTS_PATH)
    res = df.filter(pl.col("event_type") == "kill").select(["event_id", "player_id", "event_ts", "score_gained"])
    return res

def lazy_variant():
    res = (pl.scan_parquet(EVENTS_PATH).filter(pl.col("event_type") == "kill").select(["event_id", "player_id", "event_ts", "score_gained"]).collect())
    return res

def streaming_collect_variant():
    res = (pl.scan_parquet(EVENTS_PATH).filter(pl.col("event_type") == "kill").select(["event_id", "player_id", "event_ts", "score_gained"]).collect(engine="streaming"))
    return res

def streaming_sink_variant():
    res = (pl.scan_parquet(EVENTS_PATH).filter(pl.col("event_type") == "kill").select(["event_id", "player_id", "event_ts", "score_gained"]).sink_parquet(QUERY_OUTPUT_PATH))


res_eager, _ = run_benchmark(eager_variant, library_engine="polars-eager", mode="eager", query_name="q3_1_eager", rows=N_ROWS, input_path=EVENTS_PATH)

res_lazy, _ = run_benchmark(lazy_variant, library_engine="polars-lazy", mode="lazy", query_name="q3_1_lazy", rows=N_ROWS, input_path=EVENTS_PATH)

res_stream, _ = run_benchmark(streaming_collect_variant, library_engine="polars-streaming", mode="streaming", query_name="q3_1_streaming", rows=N_ROWS, input_path=EVENTS_PATH)

res_sink, _ = run_benchmark(streaming_sink_variant, library_engine="polars-sink", mode="sink", query_name="q3_1_sink", rows=N_ROWS, input_path=EVENTS_PATH)

out_size = get_file_size_mb(QUERY_OUTPUT_PATH)

res_sink["notes"] = f"Plik wynikowy sink: {out_size:.2f} MB"

[polars-eager   ] q3_1_eager                               | 0.0742s | mem +132.16 MB | input 51.36 MB
[polars-lazy    ] q3_1_lazy                                | 0.0158s | mem +17.35 MB | input 51.36 MB
[polars-streaming] q3_1_streaming                           | 0.0128s | mem +21.32 MB | input 51.36 MB
[polars-sink    ] q3_1_sink                                | 0.0344s | mem +23.04 MB | input 51.36 MB


#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [38]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.2: Identify and justify one Polars limitation.
#
# Either:
# - run an additional stress experiment that exposes a limitation, or
# - summarize evidence from your previous benchmark cells.
#
# Fill the variables below and add code if you run an extra experiment.

POLARS_LIMITATION_SCENARIO = """
Głównym ograniczeniem biblioteki Polars jest brak wsparcia dla obliczeń rozproszonych na wielu węzłach. Polars jest silnikiem działającym wyłącznie w obrębie jednej maszyny. W momencie gdy rozmiar danych wejściowych wielokrotnie przekracza fizyczny rozmiar pamięci RAM maszyny silnik Polars ulegnie awarii typu Out Of Memory. Spark dzięki architekturze Master-Worker potrafi rozdzielić pamięć i dane na wiele maszyn, a w razie braku pamięci zrzuca stany pośrednie na dysk i zapewnia odporność na awarie pojedynczych węzłów.
"""
POLARS_LIMITATION_EVIDENCE = """
Dowodem z naszych pomiarów jest to, że lokalne silniki budują całe hash-mapy dla grupowania w pamięci RAM. Na naszym zbiorze 2 000 000 wierszy czas wykonania zapytania Q3 (grupowanie o wysokiej kardynalności i COUNT DISTINCT) w Polars trwał zaledwie 0.06s dzięki przetwarzaniu in-memory. Jednak gdybyśmy przeskalowali ten zbiór 100-krotnie (np. do 200 mln wierszy), Polars wymagałby o wiele więcej RAM do alokacji tabel haszujących na jednym komputerze i uległby awarii. Spark z kolei potrafi rozłożyć operację grupowania i obliczania COUNT DISTINCT na partycje, co zapobiega awariom pamięci na pojedynczej maszynie.
"""

# YOUR OPTIONAL CODE HERE
display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)


**Polars limitation scenario**

Głównym ograniczeniem biblioteki Polars jest brak wsparcia dla obliczeń rozproszonych na wielu węzłach. Polars jest silnikiem działającym wyłącznie w obrębie jednej maszyny. W momencie gdy rozmiar danych wejściowych wielokrotnie przekracza fizyczny rozmiar pamięci RAM maszyny silnik Polars ulegnie awarii typu Out Of Memory. Spark dzięki architekturze Master-Worker potrafi rozdzielić pamięć i dane na wiele maszyn, a w razie braku pamięci zrzuca stany pośrednie na dysk i zapewnia odporność na awarie pojedynczych węzłów.

**Evidence**

Dowodem z naszych pomiarów jest to, że lokalne silniki budują całe hash-mapy dla grupowania w pamięci RAM. Na naszym zbiorze 2 000 000 wierszy czas wykonania zapytania Q3 (grupowanie o wysokiej kardynalności i COUNT DISTINCT) w Polars trwał zaledwie 0.06s dzięki przetwarzaniu in-memory. Jednak gdybyśmy przeskalowali ten zbiór 100-krotnie (np. do 200 mln wierszy), Polars wymagałby o wiele więcej RAM do alokacji tabel haszujących na jednym komputerze i uległby awarii. Spark z kolei potrafi rozłożyć operację grupowania i obliczania COUNT DISTINCT na partycje, co zapobiega awariom pamięci na pojedynczej maszynie.

#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [39]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.3: State your decision boundary.
#
# Your answer should be specific. Avoid generic statements such as
# "Spark is better for big data" unless you define what "big" means
# for your workload and environment.

DECISION_BOUNDARY = """
Na podstawie naszych pomiarów zalecamy przejście z lokalnych narzędzi takich jakPolars/DuckDB na rozproszony silnik Spark w następujących sytuacjach:
1. Rozmiar zbioru danych wejściowych przekracza około 75% fizycznej pamięci RAM na pojedynczym węźle.
2. Operacje agregujące wymagają tworzenia tabel haszujących o wysokiej kardynalności, co przekracza limit pamięci RAM pojedynczego procesu.
3. Czas cyklu uruchomienia potoku danych lokalnie przekracza limit czasu operacyjnego i wąskim gardłem staje się przepustowość dyskowa pojedynczego węzła.
"""
DECISION_EVIDENCE = """
Nasze pomiary dla 2 000 000 wierszy jednoznacznie wskazują na miażdżącą przewagę rozwiązań lokalnych pod kątem opóźnień:
- Polars Lazy (Q1): 0.0363s
- DuckDB (Q1): 0.0321s
- PySpark local (Q1): 8.1243s (ponad 250-krotnie wolniej z powodu startu JVM, serializacji danych i narzutu harmonogramowania).
Szczególnie widoczne jest to w zapytaniu Q3 (COUNT DISTINCT i grupowanie po dwóch kluczach), gdzie PySpark local potrzebował aż 23.32s ze względu na narzut fazy shuffle, podczas gdy Polars Lazy wykonał to samo zadanie w 0.068s. Dopiero gdy dane przestają mieścić się na pojedynczym dysku/pamięci RAM, skalowalność Sparka rekompensuje ten narzut startowy.
"""

display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

**Decision boundary**

Na podstawie naszych pomiarów zalecamy przejście z lokalnych narzędzi takich jakPolars/DuckDB na rozproszony silnik Spark w następujących sytuacjach:
1. Rozmiar zbioru danych wejściowych przekracza około 75% fizycznej pamięci RAM na pojedynczym węźle.
2. Operacje agregujące wymagają tworzenia tabel haszujących o wysokiej kardynalności, co przekracza limit pamięci RAM pojedynczego procesu.
3. Czas cyklu uruchomienia potoku danych lokalnie przekracza limit czasu operacyjnego i wąskim gardłem staje się przepustowość dyskowa pojedynczego węzła.

**Evidence**

Nasze pomiary dla 2 000 000 wierszy jednoznacznie wskazują na miażdżącą przewagę rozwiązań lokalnych pod kątem opóźnień:
- Polars Lazy (Q1): 0.0363s
- DuckDB (Q1): 0.0321s
- PySpark local (Q1): 8.1243s (ponad 250-krotnie wolniej z powodu startu JVM, serializacji danych i narzutu harmonogramowania).
Szczególnie widoczne jest to w zapytaniu Q3 (COUNT DISTINCT i grupowanie po dwóch kluczach), gdzie PySpark local potrzebował aż 23.32s ze względu na narzut fazy shuffle, podczas gdy Polars Lazy wykonał to samo zadanie w 0.068s. Dopiero gdy dane przestają mieścić się na pojedynczym dysku/pamięci RAM, skalowalność Sparka rekompensuje ten narzut startowy.

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [40]:
# TODO: Run selected scalability experiments and append results to benchmark_results.

import psutil
logical_cores = psutil.cpu_count(logical=True)
threads_to_test = [1, 2, 4, 8, logical_cores]

for t in threads_to_test:
    con = duckdb.connect()
    con.execute(f"SET threads TO {t}")
    
    def q1_ddb_scaled():
        return con.execute(f"""
            SELECT player_id, SUM(score_gained) AS total_score
            FROM read_parquet('{EVENTS_PATH}')
            WHERE event_type IN ('kill', 'achievement')
            GROUP BY player_id
            ORDER BY total_score DESC
            LIMIT 100
        """).df()
        
    run_benchmark(q1_ddb_scaled,library_engine="duckdb",mode=f"threads_{t}",query_name="top_players_by_score_scalability",rows=N_ROWS,notes=f"Skalowalność: limit {t} wątków", input_path=EVENTS_PATH)
    
    con.close()

[duckdb         ] top_players_by_score_scalability         | 0.0978s | mem +64.76 MB | input 51.36 MB
[duckdb         ] top_players_by_score_scalability         | 0.0587s | mem +66.39 MB | input 51.36 MB
[duckdb         ] top_players_by_score_scalability         | 0.0374s | mem +29.95 MB | input 51.36 MB
[duckdb         ] top_players_by_score_scalability         | 0.0301s | mem +30.22 MB | input 51.36 MB
[duckdb         ] top_players_by_score_scalability         | 0.0315s | mem +31.52 MB | input 51.36 MB


### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [ ]:
# gcloud storage cp -r data/phase2_26L/group_06/events.parquet gs://tbd-2026l-331459-data/events.parquet
# gcloud storage cp -r data/phase2_26L/group_06/dimension.parquet gs://tbd-2026l-331459-data/dimension.parquet
# gcloud storage cp -r data/phase2_26L/group_06/events_partitioned gs://tbd-2026l-331459-data/events_partitioned

import time
from pyspark.sql import SparkSession

spark_dataproc = (
    SparkSession.builder
    .appName("TBDPhase2DataprocBenchmark")
    .getOrCreate()
)

PROJECT_ID = "tbd-2026l-331459"
EVENTS_GCS_PATH = f"gs://{PROJECT_ID}-data/events.parquet"
DIMENSION_GCS_PATH = f"gs://{PROJECT_ID}-data/dimension.parquet"

df_events = spark_dataproc.read.parquet(EVENTS_GCS_PATH)
df_games = spark_dataproc.read.parquet(DIMENSION_GCS_PATH)

df_events.createOrReplaceTempView("events_gcs")
df_games.createOrReplaceTempView("games_gcs")

def run_dataproc_query(query_name, sql_string):
    t0 = time.perf_counter()
    result = spark_dataproc.sql(sql_string).collect()
    duration = time.perf_counter() - t0
    print(f"[Dataproc PySpark] {query_name:40s} | {duration:.4f}s | rows count: {len(result)}")
    return duration

q1_sql = """
    SELECT player_id, SUM(score_gained) AS total_score
    FROM events_gcs
    WHERE event_type IN ('kill', 'achievement')
    GROUP BY player_id
    ORDER BY total_score DESC
    LIMIT 100
"""

q2_sql = """
    SELECT g.genre, AVG(e.session_duration_s) AS avg_session_duration_s
    FROM events_gcs e
    JOIN games_gcs g ON e.game_id = g.game_id
    WHERE e.session_duration_s IS NOT NULL
      AND NOT isnan(e.session_duration_s)
    GROUP BY g.genre
    ORDER BY avg_session_duration_s DESC
"""

q3_sql = """
    SELECT event_date, country, COUNT(DISTINCT player_id) AS dau
    FROM events_gcs
    GROUP BY event_date, country
    ORDER BY event_date, country
"""

time_q1 = run_dataproc_query("top_players_by_score", q1_sql)
time_q2 = run_dataproc_query("avg_session_duration_by_game_genre", q2_sql)
time_q3 = run_dataproc_query("daily_active_players_per_country", q3_sql)


### Porównanie

| Query | Local PySpark | Dataproc PySpark | Rows Count |
| :--- | :---: | :---: | :---: |
| top_players_by_score | 11.3469s | 3.2492s | 100 |
| avg_session_duration_by_game_genre | 0.2621s | 2.5833s | 7 |
| daily_active_players_per_country | 23.8745s | 8.3873s | 630 |

**Komentarz:**
Przeniesienie obliczeń na rozproszony klaster Dataproc pozwoliło na skrócenie czasu wykonywania najbardziej wymagających zapytań (Q1 i Q3) nawet ponad trzykrotnie dzięki równoległemu przetwarzaniu partycji na wielu maszynach worker. W przypadku prostszych zadań o małej złożoności (takich jak Q2) narzut na planowanie zadań w klastrze oraz komunikację sieciową przewyższa zyski ze współbieżności, przez co wykonanie lokalne pozostaje szybsze.


*Uwaga: Wyniki dla Dataproc PySpark pochodzą z osobnego uruchomienia na dedykowanym klastrze chmurowym (w ramach Phase 1), a nie z aktualnego lokalnego wykonania całego notebooka (stąd brak numeru wykonania/execution_count w komórce powyżej).*

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [6]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
Zapytanie Q2 (złączenie events i games). W silnikach DataFrame Pandas/Polars wymaga jawnego łączenia metodami merge/join i manipulacji strukturą, natomiast w silnikach SQL DuckDB/Spark deklarujemy relację czytelnym JOIN, co pozwala optymalizatorowi zapytań na automatyczny dobór kolejności łączenia tabel.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

**Final answer 1**

Zapytanie Q2 (złączenie events i games). W silnikach DataFrame Pandas/Polars wymaga jawnego łączenia metodami merge/join i manipulacji strukturą, natomiast w silnikach SQL DuckDB/Spark deklarujemy relację czytelnym JOIN, co pozwala optymalizatorowi zapytań na automatyczny dobór kolejności łączenia tabel.

In [8]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
Zapytanie Q3 (daily_active_players_per_country). Ze względu na grupowanie po dwóch kluczach (date, country) i operację COUNT(DISTINCT player_id) o dużej liczbie unikalnych wartości, silniki są zmuszone do budowania i utrzymywania w pamięci RAM dużych, tymczasowych tabel haszujących.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

**Final answer 2**

Zapytanie Q3 (daily_active_players_per_country). Ze względu na grupowanie po dwóch kluczach (date, country) i operację COUNT(DISTINCT player_id) o dużej liczbie unikalnych wartości, silniki są zmuszone do budowania i utrzymywania w pamięci RAM dużych, tymczasowych tabel haszujących.

In [5]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
Lazy execution zmieniło ilość danych materializowanych po drodze. W wersji eager najpierw tworzony był pełny DataFrame, a dopiero później wykonywany był filtr, wybór kolumn i agregacja. W wersji lazy Polars budował plan zapytania i mógł przesunąć filtr oraz wybór kolumn bliżej źródła danych.
Dla zapytań używających tylko części kolumn lazy execution umożliwia projection pushdown, a dla zapytań z filtrem predicate pushdown. Dzięki temu silnik nie musi materializować pełnej tabeli ze wszystkimi kolumnami przed wykonaniem właściwej operacji.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

**Final answer 3**

Lazy execution zmieniło ilość danych materializowanych po drodze. W wersji eager najpierw tworzony był pełny DataFrame, a dopiero później wykonywany był filtr, wybór kolumn i agregacja. W wersji lazy Polars budował plan zapytania i mógł przesunąć filtr oraz wybór kolumn bliżej źródła danych.
Dla zapytań używających tylko części kolumn lazy execution umożliwia projection pushdown, a dla zapytań z filtrem predicate pushdown. Dzięki temu silnik nie musi materializować pełnej tabeli ze wszystkimi kolumnami przed wykonaniem właściwej operacji.

In [7]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
Strumieniowanie zmniejszyło szczytowe zużycie pamięci poprzez przetwarzanie danych w paczkach. Czas wykonania pozostał zbliżony do standardowego trybu lazy ze względu na narzut na zarządzanie wątkami strumienia.
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

**Final answer 4**

Strumieniowanie zmniejszyło szczytowe zużycie pamięci poprzez przetwarzanie danych w paczkach. Czas wykonania pozostał zbliżony do standardowego trybu lazy ze względu na narzut na zarządzanie wątkami strumienia.

In [11]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
Metoda sink_parquet(...) jest odpowiednia, gdy rozmiar wyniku zapytania jest bardzo duży i przekracza dostępną pamięć RAM, a sam wynik nie musi być materializowany ani analizowany w pamięci Pythona, lecz ma być bezpośrednio zapisany na dysk.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

**Final answer 5**

Metoda sink_parquet(...) jest odpowiednia, gdy rozmiar wyniku zapytania jest bardzo duży i przekracza dostępną pamięć RAM, a sam wynik nie musi być materializowany ani analizowany w pamięci Pythona, lecz ma być bezpośrednio zapisany na dysk.

In [41]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
Tak, lokalny Spark był znacznie wolniejszy od Polars i DuckDB na lokalnym zbiorze 2 mln wierszy. Dla Q1 czas wyniósł około 11.3 s dla PySpark local wobec około 0.03 s dla Polars/DuckDB. Wynika to z narzutu JVM oraz mechanizmu wykonania rozproszonego, który przy małych danych nie ma szans się zamortyzować.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)

**Final answer 6**

Tak, lokalny Spark był znacznie wolniejszy od Polars i DuckDB na lokalnym zbiorze 2 mln wierszy. Dla Q1 czas wyniósł około 11.3 s dla PySpark local wobec około 0.03 s dla Polars/DuckDB. Wynika to z narzutu JVM oraz mechanizmu wykonania rozproszonego, który przy małych danych nie ma szans się zamortyzować.

In [16]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
Gdy rozmiar danych przekracza ok. 75% fizycznej pamięci RAM lokalnej maszyny lub przy kosztownych agregacjach o dużej liczbie unikalnych wartości, gdzie mechanizm shuffle w Sparku rozprasza pamięć roboczą na wiele węzłów, zapobiegając błędom OOM.
"""
display_answer("Final answer 7", FINAL_ANSWER_7)

**Final answer 7**

Gdy rozmiar danych przekracza ok. 75% fizycznej pamięci RAM lokalnej maszyny lub przy kosztownych agregacjach o dużej liczbie unikalnych wartości, gdzie mechanizm shuffle w Sparku rozprasza pamięć roboczą na wiele węzłów, zapobiegając błędom OOM.

In [6]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """
Backend PyArrow zmienił głównie reprezentację typów danych, szczególnie stringów i nullable columns. Może to poprawiać pracę z Parquet i ograniczać koszt konwersji danych, ale nie gwarantuje automatycznie szybszego wykonania każdego zapytania.
W naszych wynikach PyArrow nie dał jednoznacznej przewagi czasowej we wszystkich przypadkach. Dla części zapytań wyniki były podobne albo lepsze, a dla części wariant domyślny wypadał równie dobrze lub szybciej. Korzyści z PyArrow zależą od rodzaju danych i operacji."""
display_answer("Final answer 8", FINAL_ANSWER_8)


**Final answer 8**

Backend PyArrow zmienił głównie reprezentację typów danych, szczególnie stringów i nullable columns. Może to poprawiać pracę z Parquet i ograniczać koszt konwersji danych, ale nie gwarantuje automatycznie szybszego wykonania każdego zapytania.
W naszych wynikach PyArrow nie dał jednoznacznej przewagi czasowej we wszystkich przypadkach. Dla części zapytań wyniki były podobne albo lepsze, a dla części wariant domyślny wypadał równie dobrze lub szybciej. Korzyści z PyArrow zależą od rodzaju danych i operacji.